# Assignment 2 (Alignment & Reasoning RL) on Colab A100

Runs every experiment for the writeup on a **single A100**: the policy and vLLM share the GPU
(vLLM gets ~30% of memory; it is started before the policy so it can size its cache).

**Code comes from GitHub; gitignored files come from Drive.** Each session clones the repo to local disk
(`/content/assignment2-alignment`). What git does not track is filled in like this:

| Gitignored path | Source |
|---|---|
| `data/` | copied from `MyDrive/assignment2-alignment/data` |
| `runs/` | symlink to `MyDrive/assignment2-alignment/runs` (outputs + checkpoints survive disconnects) |
| `report/figures/` | symlink to `MyDrive/assignment2-alignment/figures` |
| `models/` | downloaded from Hugging Face (cell 4) |

**One-time setup:** upload the local `data/` folder to Google Drive as `MyDrive/assignment2-alignment/data`.

**Every session:** run cells 1-4 (GPU check, clone + Drive, install, model). Then run whichever experiment cells are next.
Push code changes to GitHub first: the notebook always runs the latest commit on `BRANCH`.

**Disconnects:** GRPO and expert iteration save a checkpoint to `runs/<name>/checkpoint` (~9 GB each, on Drive, deleted
when the run finishes). If Colab drops, re-run cells 1-4 and then the same experiment cell: it resumes.
SFT and baseline runs are short and just restart.

Order: baseline → SFT (5 runs) → expert iteration (4 runs) → GRPO → plots. GRPO is by far the longest (hours).

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout
assert "A100" in gpu, f"Expected an A100 runtime, got: {gpu!r} (Runtime > Change runtime type > A100 GPU)"

## 2. Clone the repo, mount Drive, and link gitignored files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
REPO_URL = "https://github.com/deepk2001/Alignment_and_Reasoning_RL.git"
BRANCH = "main"
GITHUB_TOKEN = ""  # only needed if the repo is private (a fine-grained read-only token)
PROJECT = "/content/assignment2-alignment"         # git checkout on fast local disk
DRIVE_DIR = "/content/drive/MyDrive/assignment2-alignment"  # data/, runs/, figures/
MODEL = "/content/models/Qwen2.5-Math-1.5B"
os.environ["UV_PROJECT_ENVIRONMENT"] = "/content/a2-venv"
os.environ["UV_LINK_MODE"] = "copy"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if os.path.isdir(f"{PROJECT}/.git"):
    !git -C {PROJECT} fetch -q origin {BRANCH} && git -C {PROJECT} checkout -q {BRANCH} && git -C {PROJECT} reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} {url} {PROJECT}
!git -C {PROJECT} log --oneline -1

# Gitignored files: data is copied locally (fast reads); outputs live on Drive.
assert os.path.isdir(f"{DRIVE_DIR}/data/MATH"), f"Upload data/ to {DRIVE_DIR}/data first"
!rsync -a --exclude '.DS_Store' {DRIVE_DIR}/data/ {PROJECT}/data/
for drive_sub, local in [("runs", "runs"), ("figures", "report/figures")]:
    os.makedirs(f"{DRIVE_DIR}/{drive_sub}", exist_ok=True)
    os.makedirs(os.path.dirname(f"{PROJECT}/{local}"), exist_ok=True)
    !rm -rf {PROJECT}/{local} && ln -sfn {DRIVE_DIR}/{drive_sub} {PROJECT}/{local}

%cd {PROJECT}
!ls data runs

## 3. Install dependencies (~5 min each session)
flash-attn is skipped (slow to build); the code falls back to PyTorch SDPA attention.

In [ ]:
!pip install -q uv
!uv sync --no-install-package flash-attn 2>&1 | tail -3
!uv run --no-sync python -c "import torch, vllm, transformers; print(torch.__version__, vllm.__version__, transformers.__version__, torch.cuda.is_available())"

## 4. Download Qwen2.5-Math-1.5B to local disk
Also links it at the path the unit tests expect, so the original test suite runs unmodified.

In [ ]:
!uv run --no-sync python -c "from huggingface_hub import snapshot_download; snapshot_download('Qwen/Qwen2.5-Math-1.5B', local_dir='{MODEL}')"
!mkdir -p /data/a5-alignment/models && ln -sfn {MODEL} /data/a5-alignment/models/Qwen2.5-Math-1.5B

Optional: log to Weights & Biases as well (local `runs/*/metrics.jsonl` is always written). Set `WANDB` below and add `--wandb-project $WANDB` to the commands.

In [ ]:
WANDB = ""  # e.g. "a2-alignment"; leave empty to skip
if WANDB:
    from getpass import getpass
    os.environ["WANDB_API_KEY"] = getpass("wandb API key: ")

## 5. Unit tests (required: test_sft.py and test_grpo.py)

In [ ]:
!uv run --no-sync pytest tests/test_sft.py tests/test_grpo.py -q 2>&1 | tail -3

## 6. Zero-shot baseline — `math_baseline` (§3.2)
Generates on all 1,500 validation problems, then categorises failures for part (b).

In [ ]:
!uv run --no-sync python scripts/math_baseline.py --model {MODEL}
!uv run --no-sync python scripts/analyze_baseline.py --run baseline_r1_zero

## 7. SFT — `sft_experiment` (§4.3)
`data/sft.jsonl` is s1K-1.1 (R1 traces). The 8 questions that also appear in MATH validation are excluded automatically.
Traces are long (median ~9.7k tokens), so they are truncated to `--max-seq-len 4096` for training.

**Target: ≥15% validation accuracy for the full dataset.** If `sft_full` falls short, tune `--lr`
(try 1e-5 / 5e-5), `--num-epochs`, or `--max-seq-len`, and note it in the writeup.

In [ ]:
for n in (128, 256, 512):
    !uv run --no-sync python scripts/sft_train.py --model {MODEL} --num-examples {n} --run-name sft_n{n}

In [ ]:
!uv run --no-sync python scripts/sft_train.py --model {MODEL} --run-name sft_full

Filtered data: `data/sft_filtered.jsonl` (597 examples) is already built. Re-run the next cell only if it is missing.

In [ ]:
import os
if not os.path.exists("data/sft_filtered.jsonl"):
    !uv run --no-sync python scripts/filter_sft.py
!cat runs/sft_filter_stats.json
!uv run --no-sync python scripts/sft_train.py --model {MODEL} --data data/sft_filtered.jsonl --run-name sft_filtered

## 8. Expert iteration — `expert_iteration_experiment` (§5)
`n_ei_steps=5` on MATH train. Deliverable needs ≥2 rollout counts (G) and ≥2 SFT epoch counts, plus batch
sizes from {512, 1024, 2048}. This set of four runs varies one factor at a time from `b512_g4_e1`.
Each cell resumes automatically if Colab disconnects mid-run.

In [ ]:
!uv run --no-sync python scripts/expert_iteration.py --model {MODEL} --batch-size 512 --group-size 4 --sft-epochs 1 --run-name ei_b512_g4_e1

In [ ]:
!uv run --no-sync python scripts/expert_iteration.py --model {MODEL} --batch-size 512 --group-size 8 --sft-epochs 1 --run-name ei_b512_g8_e1

In [ ]:
!uv run --no-sync python scripts/expert_iteration.py --model {MODEL} --batch-size 512 --group-size 4 --sft-epochs 2 --run-name ei_b512_g4_e2

In [ ]:
!uv run --no-sync python scripts/expert_iteration.py --model {MODEL} --batch-size 1024 --group-size 4 --sft-epochs 1 --run-name ei_b1024_g4_e1

## 9. GRPO — `grpo_train_loop` (§7.2)
Handout starter hyperparameters: 200 steps, 256 rollouts/step (32 questions × G=8), on-policy,
`reinforce_with_baseline`, std-normalised advantages, lr 1e-5. Validation reward on 1,024 examples every 10 steps.
Checkpoints every 10 steps; just re-run this cell after a disconnect.

In [ ]:
!uv run --no-sync python scripts/grpo_train.py --model {MODEL} --run-name grpo_default

## 10. Figures for the writeup
Writes `report/figures/*.png` from whatever runs exist so far (safe to re-run any time).

In [ ]:
!uv run --no-sync --with matplotlib python scripts/make_plots.py

## 11. Results summary

In [ ]:
import glob, json
for p in sorted(glob.glob("runs/*/summary.json")):
    s = json.load(open(p))
    keys = [k for k in ("accuracy", "best_accuracy", "final_accuracy", "final_val_reward", "best_val_reward") if k in s]
    print(f"{p.split('/')[1]:<20}", {k: round(s[k], 4) for k in keys})